In [112]:
# ==========================================
# CELL 1: Setup and Functions
# ==========================================
nvars = 4 # number of variables
BR = QQ[",".join("x"+str(i) for i in range(1, nvars+1))+",z"] # polynomial ring in x1, ..., xnvars, z
# BR is a global variable
BR.inject_variables() # make it so you can use those variables
SymmetricFunctions(QQ).inject_shorthands(verbose=False) # define the bases of symmetric functions

def do_P_k(k, n):
    # we are computing p_k[s_n] = s_n[p_k]
    # evaluate at the generators gens = BR.gens()[:-1] = (x1, x2, x3, x4)
    global BR, nvars
    CR = s[1].expand(nvars).parent()
    return s[n].expand(nvars).subs({CR.gens()[i] : BR('x'+str(i+1))**k for i in range(nvars)})

@cached_function
def do_P_lambda(la, n):
    global BR, nvars
    a = BR(expand(mul(do_P_k(p, n) for p in la) * mul(BR('x'+str(i+1))-BR('x'+str(j+1)) for i in range(nvars) for j in range(i+1, nvars))))
    return sum(c * mul(BR('x'+str(i+1))^(v[i]-(nvars-i-1)) for i in range(nvars)) \
               for (v,c) in a.dict().items() if all(v[i] > v[i+1] for i in range(nvars-1)))

"""
This is the denominator that you are trying to determine
"""
@cached_function
def den_guess():
    global BR
    m1 = z*x1**5
    m2 = z*x1**3*x2**2
    m3 = -z**2 * x1**5 * x2**5 
    m4 = z * x1**4 * x2
    m5 = z**2 * x1**5 * x2**5
    m6 = -z * x1**3 * x2 * x3
    m7 = -z**3 * x1**5 * x2**5 * x3**5
    m8 = z * x1**2 * x2**2 * x3
    m9 = z**3 * x1**5 * x2**5 * x3**5
    m10 = z**2 * x1**6 * x2**2 * x3**2
    m11 = -z**2 * x1**4 * x2**3 * x3**3
    m12 = -z * x1**2 * x2**2 * x3
    m13 = z**4 * x1**5 * x2**5 * x3**5 * x4**5
    m14 = -z**2 * x1**4 * x2**4 * x3 * x4
    
    m15 = z**3 * x1**4 * x2**4 * x3**4 * x4**3
    m16 = z * x1**2 * x2 * x3 * x4
    m17 = -z**2 * x1**3 * x2**3 * x3**2 * x4**2
    m18 = -z * x1**2 * x2 * x3 * x4
    m19 = -z * x1 * x2**2 * x3 * x4
    m20 = z * x1 * x2**2 * x3 * x4
    m21 = z**2 * x1**3 * x2**3 * x3**2 * x4**2
    m22 = z**2 * x1**3 * x2**3 * x3**3 * x4
    return BR((1 - m1)*(1 - m2)*(1 - m3)**2*(1 - m4)*(1 - m5)*(1 - m6)*(1 - m7)**2*(1-m8)**2*(1-m9)*(1-m10)*(1-m11)*(1-m12)*(1-m13)*(1-m14)*(1-m15)*(1 - m16)*(1 - m17)**3*(1-m18)**2*(1-m19)**2*(1-m20)*(1-m21)*(1-m22))

# Global cache to expand the denominator exactly once
EXPANDED_DENOMINATOR = None

def get_den_expanded():
    global EXPANDED_DENOMINATOR
    if EXPANDED_DENOMINATOR is None:
        EXPANDED_DENOMINATOR = den_guess()
    return EXPANDED_DENOMINATOR

@cached_function
def den_coeff(d):
    global BR, z
    # Extremely fast native coefficient extraction (bypassing SR completely)
    return get_den_expanded().coefficient({z: d})

def calc_num(la, d):
    return sum(den_coeff(d-r) * do_P_lambda(Partition(la), r) for r in range(d+1))

Defining x1, x2, x3, x4, z


In [ ]:
# ==========================================
# CELL 2: Execution Loop
# ==========================================
out = 0

print("Warming up native denominator expansion... (takes a few seconds)")
get_den_expanded()
print("Expansion complete! Starting degrees...\n")

for d in range(0, 300):
    CC = calc_num([2,2,1], d)
    if CC:
        CC_list = list(CC)
        if len(CC_list) > 6:
            front = CC_list[:3]
            back = CC_list[-3:]
            print(d, len(CC_list), "FRONT:", front, "BACK:", back)
        else:
            print(d, len(CC_list), CC_list)
    else:
        print(d, 0, "[]")
    
    out += z**d * CC

Warming up native denominator expansion... (takes a few seconds)
Expansion complete! Starting degrees...

0 1 [(1, 1)]
1 4 [(-1, x1^4*x2), (-1, x1^3*x2*x3), (1, x1^2*x2*x3*x4), (1, x1*x2^2*x3*x4)]
2 15 FRONT: [(1, x1^8*x2^2), (-1, x1^7*x2^3), (2, x1^6*x2^4)] BACK: [(1, x1^4*x2^2*x3^2*x4^2), (1, x1^3*x2^3*x3^2*x4^2), (-1, x1^2*x2^4*x3^2*x4^2)]
3 41 FRONT: [(-1, x1^11*x2^4), (1, x1^10*x2^5), (2, x1^8*x2^7)] BACK: [(-1, x1^4*x2^5*x3^3*x4^3), (-1, x1^3*x2^6*x3^3*x4^3), (-1, x1^4*x2^4*x3^4*x4^3)]
4 82 FRONT: [(-2, x1^13*x2^7), (-1, x1^11*x2^9), (1, x1^10*x2^10)] BACK: [(-1, x1^5*x2^7*x3^4*x4^4), (-2, x1^6*x2^5*x3^5*x4^4), (-1, x1^5*x2^6*x3^5*x4^4)]
5 140 FRONT: [(1, x1^16*x2^9), (-2, x1^15*x2^10), (1, x1^14*x2^11)] BACK: [(-1, x1^7*x2^8*x3^5*x4^5), (-3, x1^7*x2^7*x3^6*x4^5), (1, x1^6*x2^8*x3^6*x4^5)]
6 206 FRONT: [(1, x1^17*x2^13), (1, x1^19*x2^10*x3), (-1, x1^18*x2^11*x3)] BACK: [(1, x1^8*x2^9*x3^7*x4^6), (1, x1^7*x2^10*x3^7*x4^6), (1, x1^8*x2^8*x3^8*x4^6)]
7 295 FRONT: [(-1, x1^21*x2^14),

In [89]:
factor(out)

x1^54*x2^40*x3^21*z^23 - x1^51*x2^39*x3^20*z^22 - x1^50*x2^39*x3^21*z^22 - x1^50*x2^37*x3^18*z^21 + 2*x1^50*x2^36*x3^19*z^21 + x1^48*x2^38*x3^19*z^21 - x1^49*x2^36*x3^20*z^21 - x1^49*x2^35*x3^21*z^21 + 2*x1^48*x2^36*x3^21*z^21 - x1^47*x2^37*x3^21*z^21 + x1^46*x2^38*x3^21*z^21 - x1^49*x2^35*x3^16*z^20 + 2*x1^48*x2^35*x3^17*z^20 + x1^47*x2^36*x3^17*z^20 + 2*x1^47*x2^35*x3^18*z^20 - x1^46*x2^36*x3^18*z^20 + x1^45*x2^37*x3^18*z^20 - x1^47*x2^34*x3^19*z^20 - x1^46*x2^35*x3^19*z^20 - 3*x1^45*x2^36*x3^19*z^20 - x1^47*x2^33*x3^20*z^20 - 3*x1^46*x2^34*x3^20*z^20 + 2*x1^45*x2^35*x3^20*z^20 + x1^43*x2^37*x3^20*z^20 + 2*x1^46*x2^33*x3^21*z^20 + x1^44*x2^35*x3^21*z^20 - x1^43*x2^36*x3^21*z^20 - x1^47*x2^33*x3^15*z^19 + x1^46*x2^34*x3^15*z^19 + 3*x1^46*x2^33*x3^16*z^19 - x1^45*x2^34*x3^16*z^19 + 2*x1^44*x2^35*x3^16*z^19 + 2*x1^46*x2^32*x3^17*z^19 - 3*x1^45*x2^33*x3^17*z^19 - 4*x1^44*x2^34*x3^17*z^19 - 3*x1^43*x2^35*x3^17*z^19 - x1^42*x2^36*x3^17*z^19 - 4*x1^45*x2^32*x3^18*z^19 + x1^44*x2^33*x3^18*z^

In [9]:
out/den_guess()

(-x1^34*x2^21*x3^5*z^12 + 2*x1^33*x2^22*x3^5*z^12 - x1^31*x2^24*x3^5*z^12 - 2*x1^33*x2^21*x3^6*z^12 - 5*x1^32*x2^22*x3^6*z^12 - 9*x1^31*x2^23*x3^6*z^12 - x1^30*x2^24*x3^6*z^12 + 3*x1^33*x2^20*x3^7*z^12 + 2*x1^32*x2^21*x3^7*z^12 - 5*x1^31*x2^22*x3^7*z^12 + 4*x1^29*x2^24*x3^7*z^12 + 2*x1^28*x2^25*x3^7*z^12 + 3*x1^32*x2^20*x3^8*z^12 - 5*x1^31*x2^21*x3^8*z^12 - 7*x1^30*x2^22*x3^8*z^12 + x1^28*x2^24*x3^8*z^12 + x1^27*x2^25*x3^8*z^12 - x1^33*x2^18*x3^9*z^12 - 7*x1^32*x2^19*x3^9*z^12 + 5*x1^31*x2^20*x3^9*z^12 + 5*x1^30*x2^21*x3^9*z^12 + x1^29*x2^22*x3^9*z^12 - 3*x1^28*x2^23*x3^9*z^12 - 6*x1^27*x2^24*x3^9*z^12 - 2*x1^26*x2^25*x3^9*z^12 - 3*x1^32*x2^18*x3^10*z^12 - 12*x1^31*x2^19*x3^10*z^12 - 5*x1^30*x2^20*x3^10*z^12 + 3*x1^29*x2^21*x3^10*z^12 + 15*x1^28*x2^22*x3^10*z^12 + 11*x1^27*x2^23*x3^10*z^12 - x1^26*x2^24*x3^10*z^12 - x1^25*x2^25*x3^10*z^12 + 2*x1^32*x2^17*x3^11*z^12 + 5*x1^31*x2^18*x3^11*z^12 - 2*x1^30*x2^19*x3^11*z^12 - 12*x1^29*x2^20*x3^11*z^12 - 4*x1^28*x2^21*x3^11*z^12 + 5*x1^26*x2^

In [7]:
# 1. Setup Environment
Sym = SymmetricFunctions(QQ)
Sym.inject_shorthands(verbose=False)
R = PolynomialRing(QQ, 'a, b, x1, x2, x3, x4, x5, z').fraction_field()
R.inject_variables()
x = R.gens()[2:-1]
a = R.gens()[0]
b = R.gens()[1]
z = R.gens()[-1]

# 2. Define Core Functions
def normalize_rational_function(Q):
    S = PolynomialRing(PolynomialRing(QQ, 'a,b'), 'x1,x2,x3,x4,x5,z')
    K = R.fraction_field()
    den = []
    factors = Q.denominator().factor()
    scalar = factors.unit()
    for (factor, exp) in factors:
        c = S(factor).constant_coefficient()
        den.append((K(factor) / c, exp))
        scalar *= c**exp
    den = Factorization(den)
    num = Q.numerator() / scalar
    return (num, den)

def CT(f, g):
    Q = f.subs({z: z / (a * b)}) * g.subs({z: a * b})
    num, den = normalize_rational_function(Q)
    PTa = MacMahonOmega(a, num, den)
    CTa = prod(PTa).subs(b=0)
    return CTa

def P(k):
    return R.one() / R.prod((1 - z * xi**k) for xi in x)

# 3. The Step-by-Step Execution
print("Computing P(1,1)...")
F_11 = CT(P(1), P(1))

print("Computing P(1,1,1)...")
F_111 = CT(P(1), F_11)

print("Computing P(1,1,1,1)...")
F_1111 = CT(P(1), F_111)

print("Computing P(2,1,1,1) [Final Step]...")
F_2111 = CT(P(2), F_1111)

# 4. Extract the Exact Denominator
print("Normalizing final rational function...")
final_num, final_den = normalize_rational_function(F_2111)

print("\n--- EXACT SYMBOLIC DENOMINATOR ---")
print(final_den)

Defining a, b, x1, x2, x3, x4, x5, z
Computing P(1,1)...


KeyboardInterrupt: 